# NumPy：時刻から発生間隔を求める

対応する本文は [NumPy](../chapters/numpy.tex) の章末課題である。最初に4個の時刻で計算の意味を確かめ、その後で課題の6個の時刻へ進む。コードセルは上からShift+Enterで実行する。最初の3問はファイルを用意せず実行できる。

## 準備：隣り合う事象の間隔を求める

観測を始めてから10秒、15秒、23秒、41秒に事象が起きたとする。間隔は `15 - 10 = 5`、`23 - 15 = 8`、`41 - 23 = 18` 秒になる。

NumPyは、数値をまとめた配列を計算するためのライブラリである。`import numpy as np` で準備し、`np.array()` で配列を作る。`np.diff()` は後ろの値から前の値を引く。

In [1]:
import numpy as np

example_times = np.array([10, 15, 23, 41])
print(np.diff(example_times))

[ 5  8 18]


表示が `[5, 8, 18]` に対応していることを確認する。次は途中の値を表示し、短い書き方でも同じ引き算をしていることを確かめる。

In [2]:
later = example_times[1:]
earlier = example_times[:-1]
print(later)
print(earlier)
print(later - earlier)

[15 23 41]
[10 15 23]
[ 5  8 18]


`later` は `[15, 23, 41]`、`earlier` は `[10, 15, 23]` となる。同じ位置の値を引くので、結果は再び `[5, 8, 18]` になる。

## 問1：課題の時刻を並べ替える

本文の6個の時刻は、意図的に順番を混ぜてある。先に小さい順へ並べないと、「次の事象までの間隔」にならない。`np.sort()` の結果だけを先に表示する。

In [3]:
original = np.array([720.0, 0.0, 60.0, 4200.0, 180.0, 780.0])
times = np.sort(original)
print(times)

[   0.   60.  180.  720.  780. 4200.]


`[0., 60., 180., 720., 780., 4200.]` になったことを確認してから、差を求める。最初の差は60秒、最後は `4200 - 780` で3420秒になる。

In [4]:
dt = np.diff(times)
print(dt)
print("時刻の形と型:", times.shape, times.dtype)
print("間隔の形と型:", dt.shape, dt.dtype)

[  60.  120.  540.   60. 3420.]
時刻の形と型: (6,) float64
間隔の形と型: (5,) float64


間隔は `[60., 120., 540., 60., 3420.]` となる。`shape` は配列の各軸の長さ、`dtype` は数値の型である。時刻は6個、間隔は5個で、どちらも `float64` になっている。

以下の `assert` は、答えが違う場合にエラーで止める確認用の命令である。`np.testing.assert_array_equal()` は配列全体を比べる。

この例の `assert_array_equal()` は、配列の形状と各要素の値を比べる。型の一致までは要求していない。小数の丸めを許して比べる後の `assert_allclose()` では、`rtol` が答えの大きさに対する割合、`atol` が絶対値での許容差となる。

In [5]:
np.testing.assert_array_equal(times, [0, 60, 180, 720, 780, 4200])
np.testing.assert_array_equal(dt, [60, 120, 540, 60, 3420])
assert times.shape == (6,) and dt.shape == (5,)

### 補足：列ごとと行ごとの合計を区別する

本文の `axis` も実際の足し算で確かめる。`axis=0` は各列の `1+4`、`2+5`、`3+6` を計算する。`axis=1` は各行の `1+2+3` と `4+5+6` を計算する。この対応は2次元配列についてのものである。

In [6]:
values = np.array([[1, 2, 3], [4, 5, 6]])
print(values.sum(axis=0))
print(values.sum(axis=1))
np.testing.assert_array_equal(values.sum(axis=0), [5, 7, 9])
np.testing.assert_array_equal(values.sum(axis=1), [6, 15])

[5 7 9]
[ 6 15]


## 問2：平均と区間ごとの件数を調べる

まず5個の間隔の平均と、0より大きい間隔の件数を求める。合計4200秒を5で割るので、平均は840秒になる。`dt > 0` は各値が0より大きいかを表し、`count_nonzero()` は真の個数を数える。

In [7]:
print("平均 [s]:", dt.mean())
print("正の間隔:", np.count_nonzero(dt > 0))
assert dt.mean() == 840 and np.count_nonzero(dt > 0) == 5

平均 [s]: 840.0
正の間隔: 5


### 指定された3区間を数える

`[60, 600)` は60を含み、600を含まない区間である。条件を `(dt >= 60) & (dt < 600)` と書く。3区間の件数は順に0、4、1になる。

In [8]:
interval_counts = []
for low, high in [(0, 60), (60, 600), (600, 3600)]:
    mask = (dt >= low) & (dt < high)
    interval_counts.append(np.count_nonzero(mask))
print(interval_counts)
assert interval_counts == [0, 4, 1]

[np.int64(0), np.int64(4), np.int64(1)]


### 同じ幅のビンに数える

ヒストグラムの各区間をビンという。次は0秒から3600秒までを600秒ずつに分ける。境界は7個、ビンは6個になる。最初のビンに4件、最後のビンに1件が入る。

`np.histogram()` は最後のビンだけ右端も含む。上の3区間の条件式と違うため、3600秒ちょうどの値で差を確かめる。

In [9]:
edges = np.arange(0.0, 3600.0 + 600.0, 600.0)
counts, bin_edges = np.histogram(dt, bins=edges)
k = counts.argmax()
print("境界:", bin_edges)
print("度数:", counts)
print("最頻ビン:", bin_edges[k:k+2])
np.testing.assert_array_equal(counts, [4, 0, 0, 0, 0, 1])
assert len(bin_edges) == len(counts) + 1
boundary = np.array([3600.0])
assert np.count_nonzero((boundary >= 600) & (boundary < 3600)) == 0
assert np.histogram(boundary, bins=edges)[0][-1] == 1

境界: [   0.  600. 1200. 1800. 2400. 3000. 3600.]
度数: [4 0 0 0 0 1]
最頻ビン: [  0. 600.]


## 問3：固定幅と対数幅を比較する

境界を30個作ると29ビンになる。対数ビンの幅は一定ではない。正の値を対象とし、0や範囲外の値が何件あるかも確認する。横軸を対数にする操作と、ビンを対数間隔にする操作は別である。

In [10]:
log_edges = np.logspace(np.log10(1), np.log10(3600), 30)
log_edges[-1] = 3600.0  # 丸め誤差によって指定した上端を除外しないようにする。
for name, bins in [("固定幅", edges), ("対数幅", log_edges)]:
    freq, _ = np.histogram(dt, bins=bins)
    print(name, "ビン数:", len(freq), "空ビン:", np.count_nonzero(freq == 0),
          "最小・最大幅:", np.diff(bins).min(), np.diff(bins).max(),
          "範囲外:", np.count_nonzero((dt < bins[0]) | (dt > bins[-1])))
    assert freq.sum() == len(dt)

固定幅 ビン数: 6 空ビン: 4 最小・最大幅: 600.0 600.0 範囲外: 0
対数幅 ビン数: 29 空ビン: 25 最小・最大幅: 0.3262674794960523 885.6154164558807 範囲外: 0


## 発展：実カタログを地域別に比べる

ここからは取得済みCSVを使う。`REAL_CSV` にそのファイルの場所を設定する。未指定のままなら、通信や取得は行わず案内を表示する。CSVの時刻を同じ基準からの秒へ変換し、並べ替えてから差を求める。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/05_numpy/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [11]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "05_numpy"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/05_numpy


In [12]:
import csv
from datetime import datetime


def intervals_from_csv(path):
    with Path(path).open(encoding="utf-8", newline="") as f:
        stamps = [datetime.fromisoformat(row["time"]) for row in csv.DictReader(f)]
    if len(stamps) < 2:
        raise ValueError("発生間隔の計算には2件以上が必要です。")
    origin = min(stamps)
    seconds = np.array([(stamp - origin).total_seconds() for stamp in stamps], dtype=np.float64)
    return np.diff(np.sort(seconds))

### 発展：日本近海と全球を比較する

日本近海を経度122–154度、緯度20–50度と定める。下のコードは取得済みの全球カタログから日本近海を抽出するので、取得期間と更新時点をそろえられる。実データのパスを設定した場合だけ実行する。地域内を抽出してから時刻差を再計算する点に注意する。全球の間隔配列を緯度で絞る操作とは異なる。

件数、期間、マグニチュード下限や種別をそろえて分布を比較する。観測範囲や検出能力が異なるので、間隔の差をそのまま地域固有の物理過程の差とは解釈しない。

In [13]:
REAL_CSV = None  # 例: ROOT / "earthquake_practice/raw/usgs_global_2024_01.csv"
if REAL_CSV is not None:
    import pandas as pd
    catalog = pd.read_csv(REAL_CSV)
    japan = catalog.loc[catalog.longitude.between(122, 154) & catalog.latitude.between(20, 50)]
    japan_path = WORK / "usgs_japan.csv"
    japan.to_csv(japan_path, index=False)
    global_dt = intervals_from_csv(REAL_CSV)
    japan_dt = intervals_from_csv(japan_path)
    print("全球:", len(global_dt), "間隔; 日本近海:", len(japan_dt), "間隔")
else:
    print("地域比較は取得済みの実カタログを指定して実行します。")

地域比較は取得済みの実カタログを指定して実行します。
